# WiFiVision — Final Report Generator

This notebook generates a complete, editable project report from the verified WiFiVision experiment results.

## Output
- `WiFiVision_Final_Report.md`
- `WiFiVision_Final_Report.txt`
- Final result tables copied into the report directory

The report deliberately uses only verified numerical results from the completed evaluation workflow.

In [7]:
# ============================================================
# 1. IMPORTS AND PROJECT PATHS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

ROOT = r"C:\Users\ROHITH KANNA S\WiFiVision"

baseline_dir = os.path.join(
    ROOT, "data", "processed", "baseline"
)

graph_dir = os.path.join(
    ROOT, "data", "processed", "topology_graph"
)

completion_dir = os.path.join(
    ROOT, "data", "processed", "final_completion"
)

report_dir = os.path.join(
    ROOT, "data", "processed", "final_report"
)

os.makedirs(report_dir, exist_ok=True)

print("Report output directory:")
print(report_dir)

Report output directory:
C:\Users\ROHITH KANNA S\WiFiVision\data\processed\final_report


In [8]:
# ============================================================
# 2. LOAD VERIFIED RESULTS
# ============================================================

baseline_errors_path = os.path.join(
    baseline_dir,
    "baseline_errors.npy"
)

graph_errors_path = os.path.join(
    graph_dir,
    "errors.npy"
)

graph_predictions_path = os.path.join(
    graph_dir,
    "predictions.npy"
)

graph_targets_path = os.path.join(
    graph_dir,
    "targets.npy"
)

required = [
    baseline_errors_path,
    graph_errors_path,
    graph_predictions_path,
    graph_targets_path
]

missing = [
    path for path in required
    if not os.path.exists(path)
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n" +
        "\n".join(missing)
    )

baseline_errors = np.load(
    baseline_errors_path
).astype(np.float32)

graph_errors = np.load(
    graph_errors_path
).astype(np.float32)

graph_predictions = np.load(
    graph_predictions_path
).astype(np.float32)

graph_targets = np.load(
    graph_targets_path
).astype(np.float32)

baseline_mpjpe = float(
    baseline_errors.mean()
)

graph_mpjpe = float(
    graph_errors.mean()
)

improvement = (
    (baseline_mpjpe - graph_mpjpe)
    / baseline_mpjpe
) * 100

print("=" * 70)
print("VERIFIED FINAL RESULTS")
print("=" * 70)

print(f"Baseline MPJPE: {baseline_mpjpe:.6f}")
print(f"Graph MPJPE   : {graph_mpjpe:.6f}")
print(f"Improvement   : {improvement:.2f}%")

VERIFIED FINAL RESULTS
Baseline MPJPE: 0.152493
Graph MPJPE   : 0.138569
Improvement   : 9.13%


In [9]:
# ============================================================
# 3. COMPUTE FINAL METRICS
# ============================================================

thresholds = [
    0.05,
    0.10,
    0.15,
    0.20
]

pck_rows = []

for threshold in thresholds:

    baseline_pck = (
        np.mean(baseline_errors <= threshold)
        * 100
    )

    graph_pck = (
        np.mean(graph_errors <= threshold)
        * 100
    )

    pck_rows.append({
        "Threshold": threshold,
        "Baseline": baseline_pck,
        "Topology Graph": graph_pck,
        "Improvement": graph_pck - baseline_pck
    })

pck_df = pd.DataFrame(
    pck_rows
)

baseline_per_joint = baseline_errors.mean(
    axis=0
)

graph_per_joint = graph_errors.mean(
    axis=0
)

joint_df = pd.DataFrame({
    "Joint": np.arange(
        1,
        len(graph_per_joint) + 1
    ),
    "Baseline Error": baseline_per_joint,
    "Topology Graph Error": graph_per_joint,
    "Improvement": (
        baseline_per_joint
        - graph_per_joint
    )
})

comparison_df = pd.DataFrame([
    {
        "Model": "Amplitude Baseline",
        "MPJPE": baseline_mpjpe
    },
    {
        "Model": "Topology Graph",
        "MPJPE": graph_mpjpe
    }
])

print("PCK RESULTS")
print(
    pck_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("\nPER-JOINT RESULTS")
print(
    joint_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

comparison_df.to_csv(
    os.path.join(
        report_dir,
        "table_model_comparison.csv"
    ),
    index=False
)

pck_df.to_csv(
    os.path.join(
        report_dir,
        "table_pck_comparison.csv"
    ),
    index=False
)

joint_df.to_csv(
    os.path.join(
        report_dir,
        "table_per_joint_comparison.csv"
    ),
    index=False
)

PCK RESULTS
 Threshold  Baseline  Topology Graph  Improvement
      0.05     16.18           27.08        10.90
      0.10     48.61           55.42         6.81
      0.15     66.39           69.21         2.82
      0.20     75.85           78.09         2.23

PER-JOINT RESULTS
 Joint  Baseline Error  Topology Graph Error  Improvement
     1        0.102118              0.089723     0.012395
     2        0.102693              0.090741     0.011952
     3        0.102164              0.087598     0.014566
     4        0.092034              0.081320     0.010714
     5        0.101706              0.089460     0.012246
     6        0.107729              0.089562     0.018167
     7        0.099609              0.084970     0.014639
     8        0.111128              0.098195     0.012933
     9        0.124490              0.115101     0.009389
    10        0.148148              0.141703     0.006445
    11        0.147076              0.138345     0.008732
    12        0.119646 

In [10]:
# ============================================================
# 4. BUILD MARKDOWN TABLES
# ============================================================

def markdown_table(
    dataframe,
    float_digits=4
):
    df = dataframe.copy()

    for column in df.columns:

        if pd.api.types.is_float_dtype(
            df[column]
        ):
            df[column] = df[column].map(
                lambda x:
                f"{x:.{float_digits}f}"
            )

    return df.to_markdown(
        index=False
    )

model_table = markdown_table(
    comparison_df,
    float_digits=6
)

pck_table = markdown_table(
    pck_df,
    float_digits=2
)

joint_table = markdown_table(
    joint_df,
    float_digits=6
)

print(model_table)

| Model              |    MPJPE |
|:-------------------|---------:|
| Amplitude Baseline | 0.152493 |
| Topology Graph     | 0.138569 |


In [11]:
# ============================================================
# 5. GENERATE FINAL REPORT
# ============================================================

report = f"""# WiFiVision: Camera-Free 2D Human Pose Estimation Using WiFi CSI and Topology-Aware Learning

## Abstract

Human pose estimation is commonly performed using cameras, but camera-based systems can raise privacy concerns and may be unreliable under poor lighting or visual occlusion. This project investigates camera-free two-dimensional human pose estimation using WiFi Channel State Information (CSI). CSI measurements were converted into temporal windows and used to predict the coordinates of 17 body joints.

The study used a subject-independent evaluation protocol. Subjects 1–7 were used for training, subject 8 was used for validation, and subjects 9–10 were reserved for testing. The reproducible amplitude-based baseline achieved an MPJPE of {baseline_mpjpe:.6f}. A topology-aware graph model achieved an MPJPE of {graph_mpjpe:.6f}, corresponding to a relative reduction of {improvement:.2f}% compared with the baseline.

A paired comparison over 972 test windows showed that the topology-aware model improved 71.40% of the windows. The mean error reduction was 0.013924, with a previously computed 95% bootstrap confidence interval of [0.012125, 0.015787]. The paired Wilcoxon signed-rank analysis indicated a statistically significant improvement. These results show that explicitly incorporating relationships among predicted body joints can improve CSI-based pose estimation within the evaluated dataset.

---

# 1. Introduction

Human pose estimation aims to infer the spatial locations of human body joints from sensor observations. Most conventional approaches rely on RGB cameras, depth sensors, or wearable devices. Although effective, these sensing modalities can be affected by lighting conditions, occlusion, device requirements, or privacy concerns.

WiFi sensing provides an alternative modality because human movement influences the wireless propagation environment. Channel State Information contains fine-grained measurements of wireless channel behavior and can potentially encode information related to human motion and body configuration.

The central objective of WiFiVision is to investigate whether temporal WiFi CSI measurements can be used to estimate a two-dimensional human pose consisting of 17 joint locations. The project also investigates whether explicitly modeling relationships among body joints improves performance compared with a conventional amplitude-based baseline.

The final research question is therefore:

> Can topology-aware modeling of relationships among predicted joints improve subject-independent WiFi CSI-based human pose estimation?

---

# 2. Problem Definition

The input consists of a temporal CSI window with shape:

`(30, 3, 114, 10)`

where the first dimension represents the temporal window and the remaining dimensions represent the CSI representation used in the project pipeline.

The prediction target is:

`(17, 2)`

representing the two-dimensional coordinates of 17 pose joints.

The task can be expressed as:

`CSI temporal window → 17 × 2 pose coordinates`

Performance is evaluated using Mean Per Joint Position Error (MPJPE):

`MPJPE = mean(||predicted_joint - target_joint||)`

Lower MPJPE indicates better pose estimation accuracy.

---

# 3. Dataset and Evaluation Protocol

The dataset contains 270 sequences with the following shapes:

- CSI/amplitude data: `(270, 297, 3, 114, 10)`
- Pose data: `(270, 297, 17, 2)`

The project uses the following subject-independent split:

- Training subjects: 1, 2, 3, 4, 5, 6, 7
- Validation subject: 8
- Test subjects: 9, 10

This split is important because the test subjects are not included during model optimization.

The temporal preprocessing configuration is:

- Window length: 30
- Stride: 15

This produced:

- Training windows: 3402
- Validation windows: 486
- Test windows: 972

---

# 4. Experimental Methodology

## 4.1 Amplitude Baseline

The reproducible baseline uses amplitude-based CSI input and predicts the 17 × 2 pose coordinates through a convolutional and temporal learning pipeline.

The baseline was regenerated and evaluated using the same test split to ensure that the final comparison was reproducible.

## 4.2 Experimental Variants

Multiple model directions were investigated during the project, including spatial modeling, temporal transformer-style modeling, phase-based input, PCA/STFT representations, spatial attention, and more advanced CSI encoders.

These experiments did not all improve upon the baseline. This is an important result: increasing architectural complexity alone did not guarantee improved generalization.

The strongest verified improvement was obtained using the topology-aware graph approach.

## 4.3 Topology Graph Model

The final model introduces explicit structure over the predicted joints.

The architecture can be summarized as:

`CSI Window → Frame CNN → GRU → Joint Projection → Graph Convolution → Graph Convolution → Self-Attention → Pose Head`

The model first extracts spatial CSI features and processes temporal information. The resulting representation is projected into joint-level embeddings. Graph operations and attention are then applied to model dependencies among the joints before predicting the final two-dimensional coordinates.

This approach differs from a direct coordinate regression baseline because the prediction pipeline explicitly incorporates relationships between joints.

---

# 5. Results

## 5.1 Main Comparison

{model_table}

The Topology Graph model achieved the best verified MPJPE in the final reproducible comparison.

The relative improvement over the amplitude baseline was:

**{improvement:.2f}%**

## 5.2 PCK Comparison

PCK values were recomputed directly from the saved test errors.

{pck_table}

The topology-aware model achieved higher joint accuracy across the evaluated error thresholds.

## 5.3 Per-Joint Analysis

{joint_table}

The per-joint analysis shows that performance is not uniform across all joints. Some joints remain substantially more difficult than others, indicating that the remaining error is partly concentrated in difficult pose configurations rather than being evenly distributed.

---

# 6. Subject-Wise Evaluation

The Topology Graph model was evaluated separately on the two held-out test subjects.

- Subject 9 MPJPE: 0.138242
- Subject 10 MPJPE: 0.138897

The mean subject MPJPE was 0.138569.

The difference between the two subject-level MPJPE values was 0.000655. Within this test split, this indicates relatively consistent performance across the two held-out subjects.

However, because only two subjects were used for the held-out test set, this should not be interpreted as evidence of universal subject generalization.

---

# 7. Error Distribution Analysis

The baseline and Topology Graph error distributions were compared over all test joints.

Previously computed distribution statistics were:

### Amplitude Baseline

- Mean: 0.152493
- Median: 0.102985
- Standard deviation: 0.137615

### Topology Graph

- Mean: 0.138569
- Median: 0.085739
- Standard deviation: 0.137933

The mean error improved by approximately 9.13%, while the median error improved by approximately 16.75%.

The stronger median reduction suggests that the model improved typical prediction quality substantially. The standard deviations remained similar, meaning that difficult high-error cases still exist.

---

# 8. Statistical Significance Analysis

The baseline and Topology Graph predictions were compared using paired test windows.

The analysis included:

- Number of paired test windows: 972
- Improved windows: 694
- Improved proportion: 71.40%
- Worsened windows: 278
- Mean error reduction: 0.013924

A Wilcoxon signed-rank test was previously applied to the paired window-level errors and produced a statistically significant result.

A bootstrap analysis produced a 95% confidence interval for the mean error reduction:

`[0.012125, 0.015787]`

Because this interval remains positive, the analysis supports the conclusion that the observed average improvement is unlikely to be explained solely by sampling variation within the evaluated test set.

---

# 9. Discussion

The experiments show that architectural complexity alone did not consistently improve CSI-based pose estimation. Several alternative representations and model designs produced worse results than the amplitude baseline.

The successful model differs because it introduces an inductive bias aligned with the output structure. Human pose coordinates are not independent variables; joints have meaningful spatial and structural relationships. The topology-aware architecture explicitly models these relationships after extracting temporal CSI features.

The final result suggests that incorporating output topology can be more useful than simply increasing encoder complexity for this dataset and evaluation setting.

The improvement should nevertheless be interpreted within the scope of the current dataset. The evaluation demonstrates improvement over the reproducible baseline on the selected held-out subjects, but it does not establish robustness across different rooms, hardware configurations, WiFi environments, or substantially larger populations.

---

# 10. Limitations

The project has several limitations.

1. **Limited held-out subjects**  
   Only subjects 9 and 10 were used in the final test split.

2. **Dataset-specific evaluation**  
   The model was evaluated under the conditions represented by the available dataset.

3. **Limited external validity**  
   The results do not establish cross-environment or cross-device robustness.

4. **Remaining difficult joints**  
   Per-joint errors show that some joints remain considerably harder to estimate.

5. **No claim of universal real-world deployment**  
   Further evaluation is required before claiming robust deployment across arbitrary environments.

---

# 11. Future Work

Future work should focus on improving the evaluation and robustness of the system rather than merely adding more architectural complexity.

Recommended directions include:

1. Larger subject-independent datasets.
2. Cross-environment evaluation.
3. Cross-device and cross-router testing.
4. Controlled investigation of calibrated amplitude and phase fusion.
5. Uncertainty estimation for ambiguous predictions.
6. Real-time latency and deployment profiling.
7. Evaluation under different human activities and motion speeds.
8. Larger-scale spatio-temporal graph architectures when sufficient training data is available.

---

# 12. Conclusion

This project investigated camera-free two-dimensional human pose estimation using WiFi CSI.

A reproducible amplitude baseline achieved an MPJPE of **{baseline_mpjpe:.6f}**. The final Topology Graph model achieved an MPJPE of **{graph_mpjpe:.6f}**, corresponding to a **{improvement:.2f}% reduction in MPJPE**.

The improvement was supported by:

- Better overall MPJPE.
- Higher PCK values across the evaluated thresholds.
- Improvement in 71.40% of paired test windows.
- A statistically significant paired Wilcoxon analysis.
- A positive bootstrap confidence interval for the mean error reduction.
- Consistent performance across the two held-out test subjects.

The results indicate that explicitly incorporating structural relationships among predicted joints is beneficial for the evaluated WiFi CSI pose estimation task.

The final conclusion is limited to the available dataset and evaluation protocol. Additional cross-subject, cross-environment, and cross-device experiments are required to establish broader real-world robustness.

---

# References

Add the actual papers, datasets, software libraries, and any external sources used in the literature review here.

Do not include references that were not actually consulted or cited in the project.
"""

report_md_path = os.path.join(
    report_dir,
    "WiFiVision_Final_Report.md"
)

report_txt_path = os.path.join(
    report_dir,
    "WiFiVision_Final_Report.txt"
)

with open(
    report_md_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

with open(
    report_txt_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

print("=" * 70)
print("FINAL REPORT GENERATED")
print("=" * 70)

print(report)

print("\nSaved:")
print(report_md_path)
print(report_txt_path)

FINAL REPORT GENERATED
# WiFiVision: Camera-Free 2D Human Pose Estimation Using WiFi CSI and Topology-Aware Learning

## Abstract

Human pose estimation is commonly performed using cameras, but camera-based systems can raise privacy concerns and may be unreliable under poor lighting or visual occlusion. This project investigates camera-free two-dimensional human pose estimation using WiFi Channel State Information (CSI). CSI measurements were converted into temporal windows and used to predict the coordinates of 17 body joints.

The study used a subject-independent evaluation protocol. Subjects 1–7 were used for training, subject 8 was used for validation, and subjects 9–10 were reserved for testing. The reproducible amplitude-based baseline achieved an MPJPE of 0.152493. A topology-aware graph model achieved an MPJPE of 0.138569, corresponding to a relative reduction of 9.13% compared with the baseline.

A paired comparison over 972 test windows showed that the topology-aware model im

In [12]:
# ============================================================
# 6. FINAL REPORT CHECKLIST
# ============================================================

checklist = [
    "Verified baseline results loaded",
    "Verified Topology Graph results loaded",
    "Final MPJPE comparison generated",
    "PCK comparison generated",
    "Per-joint comparison generated",
    "Abstract generated",
    "Introduction generated",
    "Methodology generated",
    "Experimental analysis generated",
    "Statistical analysis included",
    "Limitations included",
    "Future work included",
    "Conclusion generated",
    "Reference section left for actual consulted sources"
]

print("=" * 70)
print("FINAL REPORT CHECKLIST")
print("=" * 70)

for i, item in enumerate(
    checklist,
    start=1
):
    print(f"[{i:02d}] {item}")

print("\nReport generation complete.")

FINAL REPORT CHECKLIST
[01] Verified baseline results loaded
[02] Verified Topology Graph results loaded
[03] Final MPJPE comparison generated
[04] PCK comparison generated
[05] Per-joint comparison generated
[06] Abstract generated
[07] Introduction generated
[08] Methodology generated
[09] Experimental analysis generated
[10] Statistical analysis included
[11] Limitations included
[12] Future work included
[13] Conclusion generated
[14] Reference section left for actual consulted sources

Report generation complete.
